# Milestone 5 - Ensemble, TTA, and MAP@3 Optimization

**Label mapping**: A=0, B=1, C=2, D=3, E=4 (same as Milestone 4).

## Setup: Model Loading and Inference Helpers

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Pin seeds - the classification heads are randomly initialized, so the seed
# determines head weights and therefore the exact predictions.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Kaggle paths with local fallback
TRAIN_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_PATH  = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'
import os
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'train.csv'
    TEST_PATH  = 'test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print(f'Train: {len(train)} rows | Test: {len(test)} rows')

# Label maps
ANSWER_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
ID2ANSWER  = {v: k for k, v in ANSWER_MAP.items()}

# DeBERTa-v3-small and RoBERTa-base as 5-class classifiers
DEBERTA_NAME = 'microsoft/deberta-v3-small'
ROBERTA_NAME = 'roberta-base'

deb_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_NAME)
rob_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_NAME)

deb_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_NAME, num_labels=5)
rob_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_NAME, num_labels=5)
deb_model.eval()
rob_model.eval()
print('Both models loaded.')


Train: 2000 rows | Test: 500 rows


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight       

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Both models loaded.


### Helper Functions

- `predict_single` - run one prompt through one model, return 5-class softmax.
- `predict_batch` - batched version for full-dataset inference.
- `map_at_3` - same MAP@3 definition as Milestones 1-4.

In [2]:
def predict_single(model, tokenizer, prompt, max_length=256):
    """Return softmax probabilities (shape (5,)) for a single prompt."""
    enc = tokenizer(prompt, truncation=True, max_length=max_length,
                    return_tensors='pt')
    with torch.no_grad():
        logits = model(**enc).logits[0]   # (5,)
    return F.softmax(logits, dim=0).numpy()


def predict_batch(model, tokenizer, prompts, max_length=256, batch_size=8):
    """Batched inference. Returns (N, 5) softmax probabilities."""
    all_probs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors='pt')
        with torch.no_grad():
            logits = model(**enc).logits       # (B, 5)
        all_probs.append(F.softmax(logits, dim=1).numpy())
    return np.vstack(all_probs)


def map_at_3(truth, prediction):
    """MAP@3 for one question. prediction is a list of up to 3 letters."""
    if truth in prediction:
        return 1.0 / (prediction.index(truth) + 1)
    return 0.0

print('Helpers ready.')

Helpers ready.


---

## Q1. DeBERTa Highest-Probability Option (Row 25)

**Task**: Run DeBERTa on the prompt at row index 25 of `train.csv`, apply
softmax, and report which option gets the highest probability and what that
probability is.

In [3]:
row_25 = train.iloc[25]
prompt_25 = str(row_25['prompt'])
print(f'Row 25 prompt (first 80): {prompt_25[:80]}...')
print(f'Row 25 ground-truth answer: {row_25["answer"]}')

deb_probs_25 = predict_single(deb_model, deb_tokenizer, prompt_25)
print(f'DeBERTa probs: {dict(zip("ABCDE", deb_probs_25.round(4)))}')

deb_top_idx  = int(np.argmax(deb_probs_25))
q1_letter    = ID2ANSWER[deb_top_idx]
q1_prob      = float(deb_probs_25[deb_top_idx])
print()
print(f'ANSWER Q1: {q1_letter}, probability of {q1_letter} = {q1_prob:.4f}')

Row 25 prompt (first 80): Choose the correct answer: What is Hesse's principle of transfer in geometry? ca...
Row 25 ground-truth answer: E
DeBERTa probs: {'A': np.float16(0.2393), 'B': np.float16(0.1936), 'C': np.float16(0.2115), 'D': np.float16(0.2062), 'E': np.float16(0.1495)}

ANSWER Q1: A, probability of A = 0.2391


---

## Q2. Simple Average Ensemble - Top Option (Row 25)

**Task**: Average the DeBERTa and RoBERTa probabilities for row 25
(`[P_DeBERTa + P_RoBERTa] / 2`). Report the option with the highest averaged
probability.

In [4]:
rob_probs_25 = predict_single(rob_model, rob_tokenizer, prompt_25)
print(f'RoBERTa probs: {dict(zip("ABCDE", rob_probs_25.round(4)))}')

avg_probs_25 = (deb_probs_25 + rob_probs_25) / 2
print(f'Averaged probs: {dict(zip("ABCDE", avg_probs_25.round(4)))}')

q2_answer = ID2ANSWER[int(np.argmax(avg_probs_25))]
print()
print(f'ANSWER Q2: {q2_answer}')

RoBERTa probs: {'A': np.float32(0.1921), 'B': np.float32(0.2015), 'C': np.float32(0.2), 'D': np.float32(0.2159), 'E': np.float32(0.1904)}
Averaged probs: {'A': np.float32(0.2156), 'B': np.float32(0.1976), 'C': np.float32(0.2058), 'D': np.float32(0.2111), 'E': np.float32(0.17)}

ANSWER Q2: A


---

## Q3. Weighted Ensemble - Top Option (Row 25)

**Task**: Apply weighted probability averaging with DeBERTa weight 0.70 and
RoBERTa weight 0.30. Report the option ranked first.

In [5]:
W_DEB = 0.70
W_ROB = 0.30

weighted_probs_25 = W_DEB * deb_probs_25 + W_ROB * rob_probs_25
print(f'Weighted probs: {dict(zip("ABCDE", weighted_probs_25.round(4)))}')

q3_answer = ID2ANSWER[int(np.argmax(weighted_probs_25))]
print()
print(f'ANSWER Q3: {q3_answer}')

Weighted probs: {'A': np.float32(0.2251), 'B': np.float32(0.1961), 'C': np.float32(0.2081), 'D': np.float32(0.2092), 'E': np.float32(0.1618)}

ANSWER Q3: A


---

## Q4. Top-3 Prediction String (Row 25, Weighted Ensemble)

**Task**: Using the weighted ensemble probabilities from Q3, rank all five
options and write the top-3 as a space-separated string in Kaggle submission
format (e.g. `C A E`).

In [6]:
top3_idx_25 = np.argsort(weighted_probs_25)[::-1][:3]
top3_letters_25 = [ID2ANSWER[i] for i in top3_idx_25]
q4_answer = ' '.join(top3_letters_25)

print(f'Full ranking: {[ID2ANSWER[i] for i in np.argsort(weighted_probs_25)[::-1]]}')
print(f'Top-3 prediction: {q4_answer}')
print()
print(f'ANSWER Q4: {q4_answer}')

Full ranking: ['A', 'D', 'C', 'B', 'E']
Top-3 prediction: A D C

ANSWER Q4: A D C


---

## Q5. Run Weighted Ensemble on All Test Rows - Save submission.csv

**Task**: Run the weighted ensemble pipeline on every row of `test.csv`. Save
predictions to `submission.csv` in the format `id,prediction` where
`prediction` is the top-3 options separated by spaces. Report how many
prediction rows are in the file (excluding the header).

In [7]:
# Batch inference on the full test set
test_prompts = test['prompt'].astype(str).tolist()
print(f'Running DeBERTa on {len(test_prompts)} test prompts...')
deb_test_probs = predict_batch(deb_model, deb_tokenizer, test_prompts)
print(f'Running RoBERTa on {len(test_prompts)} test prompts...')
rob_test_probs = predict_batch(rob_model, rob_tokenizer, test_prompts)

# Weighted ensemble
ensemble_test_probs = W_DEB * deb_test_probs + W_ROB * rob_test_probs
print(f'Ensemble probs shape: {ensemble_test_probs.shape}')

# Build top-3 predictions
predictions = []
for i in range(len(test)):
    top3 = np.argsort(ensemble_test_probs[i])[::-1][:3]
    letters = [ID2ANSWER[j] for j in top3]
    predictions.append(' '.join(letters))

submission = pd.DataFrame({
    'id': test['id'].values,
    'prediction': predictions,
})
submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv')

q5_answer = len(submission)
print(f'Prediction rows (excluding header): {q5_answer}')
print()
print(f'ANSWER Q5: {q5_answer}')
print()
print(submission.head())


Running DeBERTa on 500 test prompts...
Running RoBERTa on 500 test prompts...
Ensemble probs shape: (500, 5)
Saved submission.csv
Prediction rows (excluding header): 500

ANSWER Q5: 500

   id prediction
0   1      B C A
1   2      C B A
2   3      A B C
3   4      C A B
4   5      C D B


---

## Q6. Test-Time Augmentation - Top-1 Changes (First 50 Test Rows)

**Task**: For the first 50 rows of `test.csv`, create two versions of each
prompt:
1. Original prompt
2. Instruction-augmented: prepend `"Answer the following multiple-choice
   question carefully:"` to the prompt

Run DeBERTa on both versions, average the two probability vectors, and
compare the new top-1 to the original top-1. Count how many rows produce a
different top-1 after TTA.

In [8]:
TTA_PREFIX = 'Answer the following multiple-choice question carefully: '
N_Q6 = 50

tta_diff_count = 0
for i in range(N_Q6):
    prompt = str(test.iloc[i]['prompt'])

    # Original pass
    p_orig = predict_single(deb_model, deb_tokenizer, prompt)
    top1_orig = ID2ANSWER[int(np.argmax(p_orig))]

    # Augmented pass
    aug_prompt = TTA_PREFIX + prompt
    p_aug = predict_single(deb_model, deb_tokenizer, aug_prompt)
    top1_aug = ID2ANSWER[int(np.argmax(p_aug))]

    # Average and pick the new top-1
    p_avg = (p_orig + p_aug) / 2
    top1_avg = ID2ANSWER[int(np.argmax(p_avg))]

    if top1_orig != top1_avg:
        tta_diff_count += 1

q6_answer = tta_diff_count
print(f'Rows with different Top-1 after TTA: {tta_diff_count} / {N_Q6}')
print()
print(f'ANSWER Q6: {q6_answer}')


Rows with different Top-1 after TTA: 17 / 50

ANSWER Q6: 17


---

## Q7. DeBERTa vs Weighted Ensemble - Top-1 Differences (First 100 Test Rows)

**Task**: For the first 100 rows of `test.csv`, compare the top-1 prediction
from DeBERTa alone vs the weighted ensemble. Count how many rows have
different top-1 predictions.

In [9]:
N_Q7 = 100

deb_top1_100  = [ID2ANSWER[int(np.argmax(deb_test_probs[i]))]      for i in range(N_Q7)]
ens_top1_100  = [ID2ANSWER[int(np.argmax(ensemble_test_probs[i]))] for i in range(N_Q7)]

q7_answer = sum(1 for i in range(N_Q7) if deb_top1_100[i] != ens_top1_100[i])
print(f'Rows with different Top-1 (DeBERTa vs Ensemble): {q7_answer} / {N_Q7}')
print()
print(f'ANSWER Q7: {q7_answer}')

Rows with different Top-1 (DeBERTa vs Ensemble): 3 / 100

ANSWER Q7: 3


---

## Q8. Positive Confidence Gain (First 100 Test Rows)

**Task**: For the first 100 rows of `test.csv`, record the highest class
probability (confidence) from DeBERTa alone and from the weighted ensemble.
Compute `Confidence Gain = Ensemble_Confidence - DeBERTa_Confidence` per row.
Count how many rows have a positive confidence gain (> 0).

In [10]:
pos_gain_count = 0
for i in range(N_Q7):
    deb_conf = float(np.max(deb_test_probs[i]))
    ens_conf = float(np.max(ensemble_test_probs[i]))
    gain = ens_conf - deb_conf
    if gain > 0:
        pos_gain_count += 1

q8_answer = pos_gain_count
print(f'Rows with positive confidence gain: {pos_gain_count} / {N_Q7}')
print()
print(f'ANSWER Q8: {q8_answer}')


Rows with positive confidence gain: 0 / 100

ANSWER Q8: 0


---

## Q9. Top-3 Ranking Changes After Ensembling (First 100 Test Rows)

**Task**: For the first 100 rows of `test.csv`, compare the ordered top-3
prediction strings from DeBERTa alone vs the weighted ensemble. Count how many
rows have at least one change in their ordered top-3 ranking.

In [11]:
top3_change_count = 0
for i in range(N_Q7):
    deb_top3 = [ID2ANSWER[j] for j in np.argsort(deb_test_probs[i])[::-1][:3]]
    ens_top3 = [ID2ANSWER[j] for j in np.argsort(ensemble_test_probs[i])[::-1][:3]]
    if deb_top3 != ens_top3:
        top3_change_count += 1

q9_answer = top3_change_count
print(f'Rows with at least one Top-3 change: {top3_change_count} / {N_Q7}')
print()
print(f'ANSWER Q9: {q9_answer}')

Rows with at least one Top-3 change: 27 / 100

ANSWER Q9: 27


---

## Q10. Final MAP@3 of the Weighted Ensemble (First 100 Validation Rows)

**Task**: Using the top-3 predictions from the weighted ensemble on the first
100 rows of `train.csv` (used as validation because ground-truth answers are
available), compute the MAP@3 score. Round to 4 decimal places.

In [12]:
# Inference on first 100 TRAIN rows (validation set with known answers)
train_prompts_100 = train.head(100)['prompt'].astype(str).tolist()
print('Running DeBERTa on first 100 train rows...')
deb_train_probs = predict_batch(deb_model, deb_tokenizer, train_prompts_100)
print('Running RoBERTa on first 100 train rows...')
rob_train_probs = predict_batch(rob_model, rob_tokenizer, train_prompts_100)
ensemble_train_probs = W_DEB * deb_train_probs + W_ROB * rob_train_probs

# Compute MAP@3
map3_scores = []
for i in range(100):
    truth = train.iloc[i]['answer']
    top3 = [ID2ANSWER[j] for j in np.argsort(ensemble_train_probs[i])[::-1][:3]]
    map3_scores.append(map_at_3(truth, top3))

map3_value = float(np.mean(map3_scores))
q10_answer = round(map3_value, 4)

# Per-row breakdown
hits_1 = sum(1 for s in map3_scores if s == 1.0)
hits_2 = sum(1 for s in map3_scores if s == 0.5)
hits_3 = sum(1 for s in map3_scores if abs(s - 1/3) < 1e-6)
misses = sum(1 for s in map3_scores if s == 0.0)
print(f'Rank-1 hits: {hits_1} | Rank-2 hits: {hits_2} | Rank-3 hits: {hits_3} | Misses: {misses}')
print(f'MAP@3: {map3_value}')
print()
print(f'ANSWER Q10: {q10_answer}')

Running DeBERTa on first 100 train rows...
Running RoBERTa on first 100 train rows...
Rank-1 hits: 19 | Rank-2 hits: 34 | Rank-3 hits: 18 | Misses: 29
MAP@3: 0.42

ANSWER Q10: 0.42
